In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


df=pd.read_csv("hospital.csv");
print(df.columns)

df = df.drop_duplicates()
#fill numerical and cat values 
num_cols = df.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# categorical
cat_cols = df.select_dtypes(include=['str']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

#divide data in x and y part 
X=df.drop(columns=["readmission_risk"])
Y=df["readmission_risk"]
print(X.shape[1])

# feature scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)
#train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#model one 3 layers that is hidden x-train.shape1 denote columsn and that are the input layer and 3 output classes 
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
early = EarlyStopping(monitor='val_loss', patience=5)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early],
    verbose=0
)
#plot graph 
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title("Model 1 Loss")
plt.legend()
plt.show()

loss, accuracy = model1.evaluate(X_test, y_test)
print("Model 1 Accuracy:", accuracy)

#in notebook tensor flow not run therefore not any output here 

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
#model with dropout 
model_sec = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])
#complile model with dropout
model_sec.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_sec = model_sec.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early],
    verbose=0
)


loss, accuracy2 = model_sec.evaluate(X_test, y_test,)
print("Model 2 Accuracy:", accuracy2)

In [ ]:
#K-FOLD

kf = KFold(n_splits=5, shuffle=True, random_state=42)

s1 = []
s2 = []

for train_idx, val_idx in kf.split(X):

    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1
    m1 = Sequential([
        Dense(64, activation='relu', input_shape=(X.shape[1],)),
        Dense(32, activation='relu'),
        Dense(3, activation='softmax')
    ])
    m1.compile(optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])
    
    m1.fit(X_tr, y_tr,  epochs=20,  verbose=0)
    
    s1.append(m1.evaluate(X_val, y_val, verbose=0)[1])

    # Model 2
    m2 = Sequential([
        Dense(64, activation='relu', input_shape=(X.shape[1],)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(3, activation='softmax')
    ])
    
    m2.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])
    
    m2.fit(X_tr, y_tr,epochs=20, verbose=0)
    
    s2.append(m2.evaluate(X_val, y_val, verbose=0)[1])

print("KFold Model1 Avg:", np.mean(s1))
print("KFold Model2 Avg:", np.mean(s2))

In [ ]:
clf = LogisticRegression(max_iter=100)

kf_scores = []
for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_val)
    kf_scores.append(accuracy_score(y_val, preds))

print("Sklearn Avg Accuracy:", np.mean(kf_scores))
#final reasut of all three one is simple scond is dropout and thired own is classifier s
print("\nFINAL RESULTS ")
print("NN Simple:", accuracy)
print("NN Dropout:", accuracy2)
print("Sklearn:", np.mean(kf_scores))